# 23 Train Audio Heads

**Purpose:** Train the audio situation, arousal and valence heads (with source balancing) and save them for the audio branch.

**Inputs:** Cached BEATs embeddings (notebook 20), `audio_context` and `audio_emotion` train rows.

**Outputs:** The situation, arousal and valence heads and the emotion nudge, saved to `HEADS_DIR` (`branches.audio.heads_dir` in `configs/pipeline.yaml`: `models/audio/heads/`); the decision to balance sources inside each category.

**Splits used:** train only (out-of-fold scores); val and test: not used (val is scored in notebook 24).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, f1_score

from woofdecoder.config import REPO_ROOT, load_config
from woofdecoder.taxonomy import CATEGORIES
from wooftrainer.audio_trainer.audio_heads import save_audio_heads, train_audio_heads
from wooftrainer.plots import plot_confusion_matrix

pd.set_option("display.width", 200)

In [2]:
AUDIO_CONFIG = load_config()["branches"]["audio"]  # configs/pipeline.yaml
ENCODER = AUDIO_CONFIG["encoder"]  # notebook 20
LAYER = AUDIO_CONFIG["layer"]  # notebook 20
HEADS_DIR = REPO_ROOT / AUDIO_CONFIG["heads_dir"]  # where the app reads them

## Training

Train the heads without and with source balancing and compare them.

In [3]:
results = {}
for balance in (False, True):
    trained = train_audio_heads(ENCODER, LAYER, balance_sources=balance)
    rows_ = trained.situation_rows.reset_index(drop=True)
    true_ = rows_["mapped_category"].to_numpy()
    pred_ = np.array(trained.situation.classes)[trained.situation_oof_probs.argmax(axis=1)]

    youtube = (rows_["source"] == "self_labeled").to_numpy()
    non_distressed_youtube = youtube & (true_ != "distressed_lonely")
    results[balance] = trained

    print(f"\nbalance_sources={balance}")
    print(trained.summary.round(4).to_string(index=False))
    print(
        "YouTube macro-F1 (unweighted):",
        round(f1_score(true_[youtube], pred_[youtube], average="macro"), 3),
    )
    print(
        "non-distressed YouTube called distressed:",
        f"{(pred_[non_distressed_youtube] == 'distressed_lonely').mean():.0%}",
    )
    print("Act_Env called distressed:", f"{(pred_[~youtube] == 'distressed_lonely').mean():.1%}")
    plot_confusion_matrix(
        true_[youtube],
        pred_[youtube],
        CATEGORIES,
        f"Situation head, YouTube windows, out-of-fold (balance_sources={balance})",
    )
    plt.show()

heads = results[True]  # cells below use the source-balanced version; switch if it loses


balance_sources=False
     head      C   l2  temperature  before_log_loss  before_macro_f1  after_log_loss  after_macro_f1 balance_sources
  arousal 0.0010            0.8575           0.8902           0.5716          0.8871          0.5716             NaN
  valence 0.0003            0.8308           1.0377           0.4613          1.0361          0.4613             NaN
situation 0.0003 none       0.7929           1.1544           0.5102          1.1441          0.5102           False
YouTube macro-F1 (unweighted): 0.252
non-distressed YouTube called distressed: 74%
Act_Env called distressed: 3.2%
predicted          alert_aggressive  attention_seeking  distressed_lonely  \
true                                                                        
alert_aggressive                  1                  0                  5   
attention_seeking                 1                  1                  9   
distressed_lonely                 2                  2                 40   
other    

## Per Class and Confusion Matrix (v3: source balancing)

Per-class report and confusion matrix of the source-balanced heads.

In [4]:
classes = heads.situation.classes
rows = heads.situation_rows.reset_index(drop=True)
true = rows["mapped_category"].to_numpy()
predicted = np.array(classes)[heads.situation_oof_probs.argmax(axis=1)]

print(classification_report(true, predicted, labels=classes, digits=5, zero_division=0))

plot_confusion_matrix(true, predicted, CATEGORIES, "Situation head, all train windows, out-of-fold")
plt.show()

                   precision    recall  f1-score   support

 alert_aggressive    0.21969   0.30551   0.25559      1833
attention_seeking    0.25092   0.33484   0.28687      2440
distressed_lonely    0.06294   0.78261   0.11650        46
            other    0.36328   0.38428   0.37348      3167
  playful_excited    0.54404   0.32551   0.40731      5579

         accuracy                        0.34030     13065
        macro avg    0.28817   0.42655   0.28795     13065
     weighted avg    0.39828   0.34030   0.35431     13065

rows = true category, columns = predicted
                   alert_aggressive  attention_seeking  distressed_lonely  other  playful_excited
alert_aggressive               0.31               0.25               0.05   0.25             0.15
attention_seeking              0.24               0.33               0.04   0.20             0.18
distressed_lonely              0.02               0.02               0.78   0.13             0.04
other                          0

**v2: self-labeled audio collection on all 5 categories**

precision    recall  f1-score   support  

 alert_aggressive    0.31509   0.45554   0.37252      1833  
attention_seeking    0.34264   0.35697   0.34966      2440  
distressed_lonely    0.08230   0.86957   0.15038        46  
            other    0.42683   0.45216   0.43913      3167  
  playful_excited    0.62351   0.45062   0.52315      5579  

         accuracy                        0.43567     13065  
        macro avg    0.35808   0.51697   0.36697     13065  
     weighted avg    0.47820   0.43567   0.44794     13065  

rows = true category, columns = predicted  

alert_aggressive  attention_seeking  distressed_lonely  other  playful_excited  
alert_aggressive               0.46               0.20               0.04   0.16             0.14  
attention_seeking              0.21               0.36               0.03   0.16             0.24  
distressed_lonely              0.04               0.04               0.87   0.04             0.00  
other                          0.16               0.13               0.05   0.45             0.21  
playful_excited                0.14               0.16               0.03   0.22             0.45  

**v1: self_labeled only contained distressed_lonely data**

Recall is only 0.36–0.46. About one bark in five is mistaken for each neighbouring category. (attention_seeking is the hardest: 24% become playful, 21% alert)

distressed_lonely is found too well (possible scenarios)
1. Distress really sounds distinct: whining and howling differ acoustically from barking. This is supported by the fact that Act_Env other( rest, shower, where dogs often whine) gets the most false distressed calls (4.5%).
2. A source shortcut: every distressed clip is YouTube, and nothing else in training is. The head could be recognising “YouTube audio” (compression, phone microphones, rooms) instead of distress.

Act_Env CV is split by clip with AI-generated labels: optimistic.

**v2: self-labeled audio collection on all 5 categories**

The totals already give the answer, and the shortcut is still there.
 
The Act_Env rows of the confusion matrix barely changed, so Act_Env’s false alarms are still about 420. The extra ~26 false alarms can only come from the 39 new non-distressed YouTube windows. In other words, about two thirds of YouTube alert, attention, other and playful clips are still called distressed_lonely. This is an estimate; the crosstab will show the exact split, so paste it when you have it.
 
Inside each category, the new YouTube windows (9–11) sit next to thousands of Act_Env clips. The class weights balance categories, not sources, so in the weighted data “YouTube” is still almost entirely distressed_lonely.
 
Potential Fix: balance sources inside each category, within each category that has both sources, Act_Env and YouTube each get half of that category’s weight

## By Source, per video for self-labeled clips (v2: self-labeled audio collection on all 5 categories)

Confusion matrix per source, and the self-labeled answer per video.

In [5]:
for source in ("act_env", "self_labeled"):
    keep = (rows["source"] == source).to_numpy()
    print(f"\n{source}: {keep.sum()} rows")
    plot_confusion_matrix(
        true[keep], predicted[keep], CATEGORIES, f"Situation head, {source}, out-of-fold"
    )
    plt.show()

# One answer per self-labeled video: average its windows' probabilities (what the app would see).
mine = rows["source"] == "self_labeled"
video_ids = rows.loc[mine, "clip_id"].str.split("@").str[0]
video_probs = pd.DataFrame(heads.situation_oof_probs[mine.to_numpy()], columns=classes)
video_probs["video"] = video_ids.to_numpy()
per_video = video_probs.groupby("video").mean()
per_video["predicted"] = per_video[classes].idxmax(axis=1)
print("\nper video (distressed_lonely), out-of-fold:")
print(per_video.round(2).to_string())
print(
    "\nvideos predicted distressed_lonely:",
    (per_video["predicted"] == "distressed_lonely").sum(),
    "of",
    len(per_video),
)

# Act_Env clips wrongly called distressed: the false alarms for this class.
act_env = (rows["source"] == "act_env").to_numpy()
false_alarms = (predicted[act_env] == "distressed_lonely").mean()
print(f"Act_Env clips predicted distressed_lonely: {false_alarms:.1%}")


act_env: 12980 rows
predicted          alert_aggressive  attention_seeking  distressed_lonely  \
true                                                                        
alert_aggressive                553                454                 98   
attention_seeking               585                811                 91   
other                           416                551                172   
playful_excited                 984               1426                163   

predicted          other  playful_excited  
true                                       
alert_aggressive     453              266  
attention_seeking    492              450  
other               1216              802  
playful_excited     1181             1816  

self_labeled: 85 rows
predicted          alert_aggressive  attention_seeking  distressed_lonely  \
true                                                                        
alert_aggressive                  7                  2                  0  

Distressed precision is 0.12. 3.2% of Act_Env (about 420 clips) are falsely called distressed, against 57 true windows. That’s mostly a side effect of 13,000 vs 62 rows.

## Truth & Predicted Table

Row shares for the self-labeled windows.

In [6]:
mine = (rows["source"] == "self_labeled").to_numpy()

plot_confusion_matrix(
    true[mine], predicted[mine], CATEGORIES, "Situation head, self-labeled windows, out-of-fold"
)
plt.show()

predicted          alert_aggressive  attention_seeking  distressed_lonely  other  playful_excited
true                                                                                             
alert_aggressive               0.78               0.22               0.00   0.00             0.00
attention_seeking              0.18               0.55               0.09   0.09             0.09
distressed_lonely              0.02               0.02               0.78   0.13             0.04
other                          0.00               0.20               0.60   0.10             0.10
playful_excited                0.11               0.33               0.56   0.00             0.00


## Save the heads

Save the source-balanced heads to `models/audio/`.

In [7]:
save_audio_heads(heads, HEADS_DIR)
print("saved:", sorted(path.name for path in HEADS_DIR.iterdir()))
print("situation T:", round(heads.situation.temperature, 5))
print("balance_sources:", heads.summary.loc[2, "balance_sources"])

saved: ['arousal_head.npz', 'emotion_nudge.npz', 'situation_head.npz', 'valence_head.npz']
situation T: 1.22691
balance_sources: True


- Low Temperature (near 0): Makes the model highly confident, deterministic, and conservative; it almost always picks the single most probable choice.
- High Temperature (greater than 1): Makes the model random, creative, and uniform; it increases variety and reduces over-reliance on the top prediction.

## Results and decisions

**v3: source weight balancing**

The percentaage of calling non-distressed Youtube called distressed drastically dropped from 74% to 31%, showing how the shortcut of just learning the sources is mostly broken.

Youtube source macro F1 rose from 0.252 to 0.411 after the change.

Act_Env called distressed barely changed from 3.2% to 4.0% showing that it did not have much of effect.

Although the situation row F1 score look much wors, the two rows are scored with different weights:
- Without balancing the score is ~99% Act_Env inside each category, the easier data the heaaad sees thousands of.
- With the balancing half of each category comes from 6-9 Youtube dogs, which it is being graded on harder test.

**Source check: Done (2026-11-06).**  
All distressed_lonely training audio is YouTube; no other training audio is.   
The out-of-fold result (25/27 videos, recall 0.92) may partly reflect the source rather than distress.  
To do: run the head on non-distressed YouTube clips (fusion train) once they are prepared.

There is reaal acoustic overlap: whining
- other covers resting and baths, where dogs often whine
- excited plaay is high pitched whines and yips
- distressed_lonely is mostly whining aand howling
- evidence from both ways (ex. distressed --> other 0.13) support the claim

What is left of source shortcut:
- unclear clips lean towards distressed (biggest category so far)

Both fix by:
- more Youtube clips of other aand playful_excited
- only the training data changes